<a href="https://colab.research.google.com/github/moiseevaaleksandra5757-stack/python-ai-Moiseeva-Sasha/blob/main/notebooks/week2b_read_csv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎵 Week 2B: Data Analysis — Музыкальные произведения и их авторы

**Цель**: Научиться читать CSV-файлы из репозитория GitHub в Google Colab, выполнять базовую очистку данных и подготовиться к визуализации.

**Данные** (уже загружены):
- `music_works.sparql.csv.csv` — произведения, авторы, жанры, страны, продолжительность, дата публикации, исполнитель
- `datamusic_composers.sparql.csv.csv` — биографические данные авторов (даты жизни, места, пол)

**Что мы делаем** (5 шагов):
1. Клонируем репозиторий в Colab
2. Читаем CSV-файлы в pandas DataFrame
3. Очищаем и переименовываем столбцы
4. Проверяем заполненность полей
5. Делаем быстрый обзор данных и анализ

> ⚠️ **Важно**: Столбец `work` с URL Wikidata **переименовывается в `URL`**, а не удаляется — это позволяет отличать уникальные произведения.

## 🐱 [1] Клонируем репозиторий курса в Colab

In [ ]:
# 🐱 Шаг 1. Клонируем репозиторий курса в Colab

import os

repo = "python-ai-Moiseeva-Sasha"
repo_path = f"/content/{repo}"

if not os.path.exists(repo_path):
    !git clone -q https://github.com/moiseevaaleksandra5757-stack/python-ai-Moiseeva-Sasha/tree/main/data

if os.getcwd() != repo_path:
    %cd {repo_path}

print("✅ Репозиторий готов, теперь мы работаем внутри папки", repo)

✅ Репозиторий готов, теперь мы работаем внутри папки python-ai-Moiseeva-Sasha


## 📊 Шаг 2A: Чтение CSV-файлов в pandas

Загружаем два файла из **корня репозитория**:
- **`music_works.sparql.csv.csv`** — музыкальные произведения
- **`datamusic_composers.sparql.csv.csv`** — биографии авторов

In [ ]:
# 📊 Шаг 2A. Обновление репозитория и чтение CSV-файлов

import os
import pandas as pd

# 1. 🔄 Обновляем репозиторий, чтобы получить последние изменения
print("🔄 Обновляем репозиторий с GitHub...")
!git pull -q
print("✅ Обновление завершено.\n")

# 2. 🔍 Диагностика: ищем файлы в папке data/
print("📂 Файлы в папке data/:")
data_dir = "data"
if os.path.exists(data_dir):
    all_files = os.listdir(data_dir)
    csv_files = [f for f in all_files if f.endswith('.csv')]
    print(f"   Найдено CSV-файлов: {len(csv_files)}")
    for f in csv_files:
        print(f"   - data/{f}")
else:
    print("   ❌ Папка data/ не найдена!")

# 3. 📥 Определяем точные пути к файлам (в папке data/)
file_works = None
file_authors = None

# Варианты имен для произведений (в папке data/)
possible_works = [
    "data/music_works.csv",
    "data/music_works.sparql.csv.csv"
]

# Варианты имен для авторов (в папке data/)
possible_authors = [
    "data/music_composers.csv",
    "data/datamusic_composers.sparql.csv.csv"
]

for name in possible_works:
    if os.path.exists(name):
        file_works = name
        break

for name in possible_authors:
    if os.path.exists(name):
        file_authors = name
        break

# 4. 📊 Загрузка данных
df_music = None
df_authors = None

print("\n" + "="*50)
print("📥 Загрузка данных:")

if file_works:
    try:
        df_music = pd.read_csv(file_works)
        print(f"✅ Загружено произведений: {len(df_music)} строк")
        print(f"   Файл: {file_works}")
        print(f"   Колонки: {df_music.columns.tolist()}")
    except Exception as e:
        print(f"❌ Ошибка чтения {file_works}: {e}")
else:
    print("❌ Файл с произведениями не найден!")
    print("   Проверьте файлы в папке data/")

if file_authors:
    try:
        df_authors = pd.read_csv(file_authors)
        print(f"✅ Загружено авторов: {len(df_authors)} строк")
        print(f"   Файл: {file_authors}")
        print(f"   Колонки: {df_authors.columns.tolist()}")
    except Exception as e:
        print(f"❌ Ошибка чтения {file_authors}: {e}")
else:
    print("❌ Файл с авторами не найден!")
    print("   Проверьте файлы в папке data/")

print("="*50)

🔄 Обновляем репозиторий с GitHub...
✅ Обновление завершено.

📂 Файлы в папке data/:
   Найдено CSV-файлов: 2
   - data/music_works.sparql.csv.csv
   - data/datamusic_composers.sparql.csv.csv

📥 Загрузка данных:
✅ Загружено произведений: 1260 строк
   Файл: data/music_works.sparql.csv.csv
   Колонки: ['work', 'workLabel', 'authorLabel', 'genreLabel', 'countryLabel', 'durationLabel', 'publicationDate', 'performerLabel', 'partOfLabel']
✅ Загружено авторов: 510 строк
   Файл: data/datamusic_composers.sparql.csv.csv
   Колонки: ['author', 'authorLabel', 'birthDate', 'deathDate', 'birthPlaceLabel', 'deathPlaceLabel', 'genderLabel', 'countryLabel']


## 🧹 [2B] Очистка и переименование столбцов

В вашем CSV-файле есть **технические столбцы**, которые полезны для Викиданных, но мешают простому анализу:

- Столбец `work` с URL (ссылкой на объект Wikidata) — **переименовываем в `URL`** для удобства.
  - Это важно: одно произведение может иметь нескольких авторов = несколько строк.
  - **Без `URL` нельзя подсчитать реальное количество произведений.**
- Столбцы `workLabel`, `authorLabel`, `genreLabel`, `countryLabel`, `durationLabel` содержат читаемые подписи (названия).

В этом шаге мы:
- **переименовываем** столбец с URL Wikidata (`work` → `URL`);
- переименовываем столбцы, убирая постфикс `Label`:
  - `workLabel` → `work`
  - `authorLabel` → `author`
  - `genreLabel` → `genre`
  - `countryLabel` → `country`
  - `durationLabel` → `duration`
- приводим столбец `duration` к числовому типу.

⚠️ **Важно про `duration`**: пустое значение означает «продолжительность неизвестна», а **не** «0 секунд». Поэтому мы **не заполняем пропуски нулями**, а оставляем `NaN`.

🔁 **Идемпотентность**: код проверяет текущую структуру данных и пропускает шаги, если они уже выполнены.

In [ ]:
# 🧹 Шаг 2B. Очистка и переименование столбцов для df_music
# 🔁 Идемпотентно | ⚠️ ИСПРАВЛЕНО: work → URL (не удаляем!), duration без fillna(0)

def clean_music_dataframe(df):
    """
    Очищает и переименовывает столбцы в DataFrame.
    work → URL (не удаляем!), duration без fillna(0)
    """
    # === Проверка: DataFrame пустой? ===
    if df is None or len(df) == 0:
        print("❌ Ошибка: DataFrame пустой!")
        return df

    # === Проверка: данные уже очищены? ===
    # Ожидаемые "чистые" колонки (включая URL!)
    expected_clean_cols = {"URL", "work", "author", "genre", "country", "duration"}
    current_cols = set(df.columns)

    if expected_clean_cols.issubset(current_cols):
        label_cols = [c for c in current_cols if c.endswith("Label")]
        if not label_cols:
            print("⏭️ df_music уже очищен")
            return df

    df_clean = df.copy()

    # 1) work → URL (ПЕРЕИМЕНОВЫВАЕМ, не удаляем!)
    if "work" in df_clean.columns:
        df_clean = df_clean.rename(columns={"work": "URL"})
        print("✏️ 'work' переименован в 'URL'")

    # 2) *Label → без постфикса
    rename_map = {
        "workLabel": "work",
        "authorLabel": "author",
        "genreLabel": "genre",
        "countryLabel": "country",
        "durationLabel": "duration",
    }
    existing = {k: v for k, v in rename_map.items() if k in df_clean.columns}
    if existing:
        df_clean = df_clean.rename(columns=existing)
        print(f"✏️ Переименованы: {list(existing.keys())}")

    # 3) duration → numeric (БЕЗ fillna(0)!)
    if "duration" in df_clean.columns:
        df_clean["duration"] = pd.to_numeric(
            df_clean["duration"], errors="coerce"
        )  # ← NaN остаётся NaN
        print("🔢 'duration' преобразован (NaN сохранён)")

    return df_clean

# === Применяем ===
df_music = clean_music_dataframe(df_music)

print("\n✅ Данные готовы к анализу")
print(f"📋 Колонки: {df_music.columns.tolist()}")

# ⚠️ СТАТИСТИКА: строки vs уникальные произведения
print("\n📊 ВАЖНАЯ СТАТИСТИКА:")
print(f"   Всего строк: {len(df_music)}")
print(f"   Уникальных произведений (URL): {df_music['URL'].nunique()}")
print(f"   Уникальных авторов: {df_music['author'].nunique()}")
print(f"\n💡 Почему строк > произведений?")
print(f"   Одно произведение может иметь нескольких авторов")
print(f"   → {len(df_music)/df_music['URL'].nunique():.2f} строк на произведение в среднем")

✏️ 'work' переименован в 'URL'
✏️ Переименованы: ['workLabel', 'authorLabel', 'genreLabel', 'countryLabel', 'durationLabel']
🔢 'duration' преобразован (NaN сохранён)

✅ Данные готовы к анализу
📋 Колонки: ['URL', 'work', 'author', 'genre', 'country', 'duration', 'publicationDate', 'performerLabel', 'partOfLabel']

📊 ВАЖНАЯ СТАТИСТИКА:
   Всего строк: 1260
   Уникальных произведений (URL): 592
   Уникальных авторов: 426

💡 Почему строк > произведений?
   Одно произведение может иметь нескольких авторов
   → 2.13 строк на произведение в среднем


## 🔍 [3] Обзор данных: структура и первые строки

Сделаем короткий обзор DataFrame `df_music`:

- посмотрим размер таблицы (`shape`);
- выведем список столбцов;
- **сравним количество строк и уникальных произведений (URL)**;
- посмотрим первые несколько строки;
- посчитаем базовую статистику по числовым столбцам (`duration`).

⚠️ **Важно**: количество строк может превышать количество уникальных произведений, потому что:
- Одно произведение может иметь **нескольких авторов** (композитор + поэт)
- Одно произведение может быть **в нескольких жанрах**
- Каждая комбинация `произведение + автор + жанр` создаёт отдельную строку

Для удобства напишем функцию `show_info(df, name)`.

🔁 **Идемпотентность**: функция безопасно работает при повторных запусках — проверяет наличие столбцов перед выводом статистики.

In [ ]:
# 🔍 Шаг 3. Обзор данных: структура и первые строки
# 🔁 Идемпотентно: безопасно при повторных запусках

def show_info(df, name, n=5):
    """
    Краткий обзор DataFrame: имя, размер, список столбцов, первые строки и статистика.
    🔁 Идемпотентно: проверяет наличие столбцов перед операциями.
    """
    print(f"\n📊 {name}")
    print("=" * 60)
    print("📏 Размер:", df.shape, f"({df.shape[0]} строк, {df.shape[1]} столбцов)")
    print("📋 Столбцы:", ", ".join(df.columns))

    # 🔗 Статистика по URL (уникальные произведения) — требование преподавателя
    if "URL" in df.columns:
        print(f"\n🔗 Уникальных произведений (URL): {df['URL'].nunique()}")
        print(f"   → {len(df)/df['URL'].nunique():.2f} строк на произведение в среднем")
        print(f"   💡 Строки > произведений, т.к. одно произведение может иметь")
        print(f"      нескольких авторов или быть в нескольких жанрах")

    print("\n🔍 Первые строки:")
    display(df.head(n))

    # 🔢 Базовая статистика только по числовым столбцам
    numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
    if numeric_cols:
        print(f"\n📈 Статистика по числовым столбцам ({', '.join(numeric_cols)}):")
        display(df[numeric_cols].describe())
    else:
        print("\n⚠️ Нет числовых столбцов для статистики")

    # ❓ Информация о пропусках
    null_counts = df.isnull().sum()
    if null_counts.any():
        print("\n❓ Пропущенные значения:")
        print(null_counts[null_counts[null_counts > 0].index].to_string())
    else:
        print("\n✅ Пропущенных значений нет")

# === Применяем функцию к вашему DataFrame ===
show_info(df_music, "Музыкальные произведения: авторы, жанры, страны, продолжительность (df_music)")


📊 Музыкальные произведения: авторы, жанры, страны, продолжительность (df_music)
📏 Размер: (1260, 9) (1260 строк, 9 столбцов)
📋 Столбцы: URL, work, author, genre, country, duration, publicationDate, performerLabel, partOfLabel

🔗 Уникальных произведений (URL): 592
   → 2.13 строк на произведение в среднем
   💡 Строки > произведений, т.к. одно произведение может иметь
      нескольких авторов или быть в нескольких жанрах

🔍 Первые строки:


,URL,work,author,genre,country,duration,publicationDate,performerLabel,partOfLabel
0,http://www.wikidata.org/entity/Q2600426,Arms of Mary,Iain Sutherland,поп-музыка,NaN,NaN,1992-01-01T00:00:00Z,Piet Veerman,NaN
1,http://www.wikidata.org/entity/Q2600426,Arms of Mary,Iain Sutherland,поп-музыка,NaN,NaN,1992-01-01T00:00:00Z,The Sutherland Brothers,NaN
2,http://www.wikidata.org/entity/Q11794549,"О, мой розмарин",Wacław Denhoff-Czarnocki,NaN,Польша,NaN,1915-01-01T00:00:00Z,NaN,NaN
3,http://www.wikidata.org/entity/Q5691988,Heart Half Empty,Дезмонд Чайлд,кантри,США,NaN,2003-01-01T00:00:00Z,Ty Herndon,NaN
4,http://www.wikidata.org/entity/Q5691988,Heart Half Empty,Дезмонд Чайлд,кантри,Венгрия,NaN,2003-01-01T00:00:00Z,Стефани Бентли,NaN



📈 Статистика по числовым столбцам (duration):


,duration
count,139.000000
mean,201.167866
std,59.879092
min,4.583333
25%,161.000000
50%,203.000000
75%,236.000000
max,323.000000



❓ Пропущенные значения:
genre               525
country             200
duration           1121
publicationDate     414
performerLabel      487
partOfLabel        1073


## 📊 [5A] Топ-10 самых продуктивных авторов

В этом блоке мы детально анализируем вклад каждого автора:

- **Базовый рейтинг**: сколько произведений каждого автора представлено в датасете;
- **Расширенные метрики** для топ-10 авторов:
  - 🎭 Уникальных жанров: в скольких разных жанрах работал автор;
  - 🌍 Уникальных стран: в скольких странах представлены его работы;
  - ⏱️ Средняя и медианная длительность: характерная продолжительность произведений;
  - 🔗 Всего комбинаций `жанр + страна`: насколько разнообразен репертуар.

🔁 **Идемпотентность**: код проверяет наличие столбцов и корректно обрабатывает пропуски.

---

## 🌍 [5B] Страна-жанр матрица: география музыкальных стилей

Здесь мы исследуем связь между страной и жанром:

- **Кросс-таблица**: сколько произведений каждого жанра создано в каждой стране;
- **Доминирующий жанр по странам**: какой жанр чаще всего встречается в каждой стране;
- **Глобальность жанров**: в скольких странах представлен каждый жанр (Топ-10 "международных" жанров);
- **Разнообразие по странам**: в какой стране представлено наибольшее количество разных жанров;
- **Локальные жанры**: какие жанры встречаются только в одной стране (Топ-10).

💡 Эта матрица помогает ответить на вопросы:
- "Какие жанры характерны для Великобритании?"
- "Есть ли жанры, которые встречаются по всему миру?"
- "В какой стране самое разнообразное музыкальное наследие?"

🔁 **Идемпотентность**: все расчёты безопасны при повторных запусках и корректно работают с пропусками.

In [ ]:
# 📊 Шаг 5. Анализ авторов и страна-жанр матрица
# 🔁 Идемпотентно: проверяет наличие столбцов перед выполнением

# === Проверка: есть ли нужные колонки? ===
required_cols = ["author", "genre", "country", "duration", "URL"]
missing_cols = [c for c in required_cols if c not in df_music.columns]

if missing_cols:
    print(f"❌ Ошибка: отсутствуют колонки: {missing_cols}")
    print("   Запустите сначала Шаг 3 (очистка данных)")
else:

    # === 📊 ШАГ 5A: Топ-10 самых продуктивных авторов ===
    print("🎵 Анализ продуктивности авторов")
    print("=" * 70)

    # Базовый рейтинг
    author_counts = df_music["author"].value_counts().head(10)

    print("\n🏆 Топ-10 самых продуктивных авторов (по числу произведений):")
    for rank, (author, count) in enumerate(author_counts.items(), 1):
        print(f"   {rank}. {author}: {count} произведений")

    # Расширенные метрики
    print("\n🔍 Детальная статистика для топ-10 авторов:")
    print("-" * 70)

    def get_author_stats(group):
        return pd.Series({
            "works_count": len(group),
            "unique_genres": group["genre"].nunique(),
            "unique_countries": group["country"].nunique(),
            "avg_duration": group["duration"].mean(),
            "median_duration": group["duration"].median(),
            "genre_country_combos": group.groupby(["genre", "country"]).ngroups
        })

    author_stats = df_music.groupby("author").apply(get_author_stats).reset_index()
    top_authors = author_stats.nlargest(10, "works_count")[
        ["author", "works_count", "unique_genres", "unique_countries",
         "avg_duration", "median_duration", "genre_country_combos"]
    ].round(2)

    for _, row in top_authors.iterrows():
        print(f"\n✍️  {row['author']}")
        print(f"   📚 Произведений: {int(row['works_count'])}")
        print(f"   🎭 Жанров: {int(row['unique_genres'])} | 🌍 Стран: {int(row['unique_countries'])}")
        avg_dur = row['avg_duration'] if pd.notna(row['avg_duration']) else 0
        med_dur = row['median_duration'] if pd.notna(row['median_duration']) else 0
        print(f"   ⏱️ Длительность: ср. {avg_dur:.1f}, медиана {med_dur:.1f}")
        print(f"   🔗 Уникальных комбинаций жанр+страна: {int(row['genre_country_combos'])}")

    print("\n" + "=" * 70)


    # === 🌍 ШАГ 5B: Страна-жанр матрица ===
    print("\n🌍 Анализ связи страны и жанра")
    print("=" * 70)

    # 1. Кросс-таблица
    print("\n📋 Кросс-таблица: количество произведений по странам и жанрам")
    print("(показываем только ненулевые ячейки, топ-20 по частоте)")

    crosstab = pd.crosstab(
        df_music["country"].fillna("Unknown"),
        df_music["genre"].fillna("Unknown")
    )
    crosstab_long = crosstab.stack().reset_index(name="count")
    crosstab_long = crosstab_long[crosstab_long["count"] > 0].nlargest(20, "count")

    print(f"\n{'Страна':<25} {'Жанр':<20} {'Кол-во':>8}")
    print("-" * 55)
    for _, row in crosstab_long.iterrows():
        print(f"{row['country']:<25} {row['genre']:<20} {row['count']:>8}")

    # 2. Доминирующий жанр по странам
    print("\n🎯 Доминирующий жанр в каждой стране (Топ-10 стран по числу записей):")

    def get_dominant_genre(x):
        clean = x.dropna()
        return clean.value_counts().idxmax() if not clean.empty else "Unknown"

    dominant_genres = df_music.groupby("country")["genre"].agg(get_dominant_genre).reset_index(name="dominant_genre")
    country_counts = df_music["country"].value_counts().head(10).reset_index()
    country_counts.columns = ["country", "total_records"]
    top_countries_dominant = country_counts.merge(dominant_genres, on="country", how="left")

    for _, row in top_countries_dominant.iterrows():
        print(f"   • {row['country']:<25} → {row['dominant_genre']} ({row['total_records']} записей)")

    # 3. Глобальность жанров (ИСПРАВЛЕНО: используем URL для уникальных произведений)
    print("\n🌐 Топ-10 самых 'глобальных' жанров (по числу стран):")

    genre_countries = df_music.groupby("genre")["country"].nunique().sort_values(ascending=False).head(10)
    for rank, (genre, n_countries) in enumerate(genre_countries.items(), 1):
        # ✅ ИСПРАВЛЕНО: используем URL для подсчёта уникальных произведений
        total_works = df_music[df_music["genre"] == genre]["URL"].nunique()
        print(f"   {rank}. {genre:<20} → в {n_countries} странах ({total_works} произведений)")

    # 4. Разнообразие жанров по странам (ИСПРАВЛЕНО: используем URL)
    print("\n🎨 Топ-10 стран по жанровому разнообразию (число уникальных жанров):")

    country_genre_diversity = df_music.groupby("country")["genre"].nunique().sort_values(ascending=False).head(10)
    for rank, (country, n_genres) in enumerate(country_genre_diversity.items(), 1):
        # ✅ ИСПРАВЛЕНО: используем URL для подсчёта уникальных произведений
        total_works = df_music[df_music["country"] == country]["URL"].nunique()
        print(f"   {rank}. {country:<25} → {n_genres} жанров ({total_works} произведений)")

    # 5. Локальные жанры
    print("\n🔒 Топ-10 'локальных' жанров (встречаются только в 1 стране):")

    local_genres = df_music.groupby("genre")["country"].nunique()
    local_genres = local_genres[local_genres == 1].sort_values(ascending=False)

    if len(local_genres) > 0:
        for rank, genre in enumerate(local_genres.head(10).index, 1):
            country = df_music[df_music["genre"] == genre]["country"].mode().iloc[0] if not df_music[df_music["genre"] == genre]["country"].mode().empty else "Unknown"
            count = len(df_music[df_music["genre"] == genre])
            print(f"   {rank}. {genre:<25} → только в {country} ({count} произведений)")
    else:
        print("   ⚠️ Нет жанров, встречающихся только в одной стране")

    print("\n✅ Анализ страна-жанр матрицы завершён")

🎵 Анализ продуктивности авторов

🏆 Топ-10 самых продуктивных авторов (по числу произведений):
   1. Франтишек Ладислав Челаковский: 69 произведений
   2. Роберт Бёрнс: 51 произведений
   3. Леош Яначек: 42 произведений
   4. Lotar Olias: 36 произведений
   5. Людвик Куба: 33 произведений
   6. Walter Rothenburg: 32 произведений
   7. Вилли Девиль: 24 произведений
   8. Герберт Крецмер: 24 произведений
   9. Werner Theunissen: 21 произведений
   10. Франтишек Бартош: 21 произведений

🔍 Детальная статистика для топ-10 авторов:
----------------------------------------------------------------------

✍️  Франтишек Ладислав Челаковский
   📚 Произведений: 69
   🎭 Жанров: 0 | 🌍 Стран: 1
   ⏱️ Длительность: ср. 0.0, медиана 0.0
   🔗 Уникальных комбинаций жанр+страна: 0

✍️  Роберт Бёрнс
   📚 Произведений: 51
   🎭 Жанров: 0 | 🌍 Стран: 1
   ⏱️ Длительность: ср. 0.0, медиана 0.0
   🔗 Уникальных комбинаций жанр+страна: 0

✍️  Леош Яначек
   📚 Произведений: 42
   🎭 Жанров: 0 | 🌍 Стран: 2
   ⏱️ Длите

/tmp/ipykernel_7524/931007843.py:38: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  author_stats = df_music.groupby("author").apply(get_author_stats).reset_index()


## 📝 Summary — Что мы сделали в этом ноутбуке (Week 2B)

✅ **Клонировали репозиторий GitHub в Colab**
   - Репозиторий: `python-ai-Moiseeva-Sasha`
   - Рабочая директория: `/content/python-ai-Moiseeva-Sasha`

✅ **Прочитали CSV-файлы из репозитория**
   - `music_works.sparql.csv.csv` — музыкальные произведения (авторы, жанры, страны, длительность, дата публикации, исполнитель)
   - `datamusic_composers.sparql.csv.csv` — биографии авторов (даты жизни, места рождения/смерти, пол)

✅ **Очистили и переименовали столбцы**
   - `work` (URL Wikidata) → `URL` (сохранён для идентификации уникальных произведений)
   - `*Label` → короткие имена: `work`, `author`, `genre`, `country`, `duration`
   - `duration` преобразован в числовой тип (пропуски оставлены как `NaN`, а не заменены на 0)

✅ **Проверили структуру данных**
   - Размер таблицы: количество строк и столбцов
   - Список колонок и их типы
   - Первые строки для визуальной проверки
   - Статистика по числовым столбцам

✅ **Выполнили быструю валидацию**
   - Количество уникальных произведений (по `URL`), авторов, жанров, стран
   - Топ-5 стран и Топ-10 жанров по числу записей
   - Топ-10 самых продуктивных авторов
   - Проверка заполненности полей: `genre`, `country`, `duration`
   - Анализ страна-жанр матрицы: доминирующие, глобальные и локальные жанры

✅ **Подготовили данные для следующего этапа**
   - Теперь у нас есть аккуратные, проверенные таблицы `df_music` и `df_authors`
   - Столбец `URL` позволяет корректно считать уникальные произведения
   - Пропуски в `duration` сохранены как `NaN` (неизвестная длительность ≠ 0)


